In [ ]:
pip install youtube-transcript-api


In [ ]:
pip install langchain-groq

In [83]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()




True

In [56]:
# Document Loader


In [57]:
yt_api=YouTubeTranscriptApi()
# paste the id of video form youtube
video_id='0XStJayUrUY'  
transcripts=yt_api.fetch(video_id)

In [58]:
transcripts_text=" ".join(doc.text for doc in transcripts)

In [59]:
print(transcripts_text)

If I put you in a room with 10 entrepreneurs, one of them in next decade will become a billionaire. The other nine won't. [music] What are the other nine people are doing wrong versus one who did right? A lot of people are starting companies because it's sexy, opportunistic, but not really for the right reasons. So the main reason a really good company is created is is there a pain point to solve that the founder personally had for someone close to them. So those founders are the best of best because you have a clear purpose. These founders and founding teams we believe outperform the let's say opportunistic founders 9 is to1. [music] >> Sasha Mi Chundani founder and managing partner at K Capital and co-founder of Mumbai Angels. If you want to understand what a top investor really bets on and how great companies are actually built, this episode is for you. [music] Tell me who are the top three or four people you need in a founding team and what kind of skills they should have so that p

In [60]:
# Text Splitting

In [61]:
spiltter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=spiltter.split_text(transcripts_text)

In [62]:
len(chunks)

276

In [63]:
## Embedding Model

In [64]:
pip install -U sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [65]:
embedding=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1800.57it/s]


In [66]:
## vector Store

In [67]:
vector_store=FAISS.from_texts(chunks,embedding)

In [68]:
vector_store.similarity_search(query="how to start a startup ")

[Document(id='03e8d854-9407-4bfb-93d9-3ca90a91a9d7', metadata={}, page_content="be deaths. That's part of our lives. But that category AI and tech combo is very very very exciting. It's huge and as VCs we are very excited to be part of that journey. >> Tell me one specific startup idea or an opportunity which a young person can take it from you and build it today. Well, now take I'll give you an example of one company we just did recently at the beginning of the year, Super Living, right? It's young founding team, amazing founders. They had found a market fit because they"),
 Document(id='a7b232d1-2a5c-4376-9c67-e5967a002007', metadata={}, page_content="three or four people you need in a founding team and what kind of skills they should have so that people can build their ideal founding team. You need someone who can sell. Entrepreneurship is selling. There is no business then someone who can actually build the product. Can someone be a great salesperson who or she's not only selling t

In [69]:
## Retriever


In [70]:
retriever=vector_store.as_retriever(search_kwargs={"k":4})

In [71]:
retriever.invoke("How to start the company")

[Document(id='a7b232d1-2a5c-4376-9c67-e5967a002007', metadata={}, page_content="three or four people you need in a founding team and what kind of skills they should have so that people can build their ideal founding team. You need someone who can sell. Entrepreneurship is selling. There is no business then someone who can actually build the product. Can someone be a great salesperson who or she's not only selling the company but raising capital getting in front of people like me? >> So you need a seller and a builder. Yeah. >> You [music] said founder is everything. How do"),
 Document(id='07d9405d-3222-4106-bca1-14b1e381c882', metadata={}, page_content="I was at Nokia venture/blun if I was sitting in California office all those young founders used to come to us they would put on the first page Raj why we failed and what we failed >> nice first page no nothing what I just said I started XY Z company these are the reasons and India because of stereotype and this un unfortunate fear of f

In [72]:
## Augmentation

In [73]:


prompt = PromptTemplate(
    template="""You are an intelligent assistant that answers questions strictly based on the provided YouTube video transcript.

INSTRUCTIONS:
- Answer ONLY using the information from the context below.
- If the answer is not in the context, say: "I don't know "
- Do NOT make up facts, examples, or timestamps.
- Be concise but complete. Use bullet points for multi-part answers.
- If quoting the speaker, wrap the quote in quotation marks.
- Mention approximate timestamps if they appear in the context.

CONTEXT FROM VIDEO TRANSCRIPT:
{context}

USER QUESTION:
{question}

ANSWER:""",
    input_variables=["context", "question"]
)

In [74]:
question="If how to start the company has been spoken in this video ?"

In [75]:
context=retriever.invoke(question)

In [76]:
context=" ".join(i.page_content for i in context)

In [77]:
final_prompt=prompt.invoke({'question':question,'context':context})

In [78]:
## Genration

In [89]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

In [90]:
answer=llm.invoke(final_prompt)
print(answer.content)

Yes – the transcript includes several key points that explain how to start a company:

- **Identify the problem** – “Second page is problem.”  
- **Show the solution** – “One slide on product if you can show how it works on a video.”  
- **Demonstrate traction** – “Click about traction… competition slide.”  
- **Explain funding needs** – “One slide on how much money you want to raise. We raising 1 million.”  
- **Understand the customer** – “How much do you truly understand your customer… have they really spent time with the customers and seen the customer's pain point?”  

These elements together outline the basic steps for launching a startup.
